## simple_triton example

This notebook illustrates how to use the simple_triton package to perform inference on tiles from a whole-slide image using a Triton inference server.

Notes for running:
- See https://github.com/PathologyDataScience/simple_triton for details on launching the triton server container and mounting the model repository notebook
- This notebook requires installation of `mil`, `glimr`, and `histomics_stream`
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount your model repository directory to the triton container
- Load the model (below)

In [5]:
# install large_image with tile sources
!pip install ../../histomics_stream 'large_image[tiff]' \
  scikit_image --find-links https://girder.github.io/large_image_wheels

# install simple_triton
!pip install ../../simple_triton

# install ray tune dependencies and mil
!pip install pyarrow
!pip install tabulate
!pip install ray
!pip install ../../glimr
!pip install ../../mil

Defaulting to user installation because normal site-packages is not writeable
Looking in links: https://girder.github.io/large_image_wheels
Processing /home/lac5440/histomics_stream
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


  Created wheel for histomics_stream: filename=histomics_stream-2.3.0-py3-none-any.whl size=25525 sha256=4812e3f41856894e9ba9101083925fe5beea49c07ec08d2e7651a87879d676f5
  Stored in directory: /tmp/pip-ephem-wheel-cache-6f_o61x8/wheels/25/fc/79/449f34c4ebbb920446a89d738a01f9b5303acdea7eb13d8c9a
Successfully built histomics_stream
  Attempting uninstall: histomics_stream
    Found existing installation: histomics_stream 2.3.0
    Uninstalling histomics_stream-2.3.0:
      Successfully uninstalled histomics_stream-2.3.0
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Defaulting to user installation because normal site-packages is not writeable
Processing /home/lac5440/simple_triton
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


  Created wheel for simple-triton: filename=simple_triton-0.1.dev418+g6eb2c6b.d20230920-py3-none-any.whl size=29435 sha256=c4ec3fe217e56f606c95f991ce3af5a482fb48c261c85e2aa4677b4e92d90ef5
  Stored in directory: /tmp/pip-ephem-wheel-cache-f2h5xwsl/wheels/98/c4/7b/2a3753547f3062a82c82afa48a94ffe0cd0b85863a054c4703
Successfully built simple-triton
  Attempting uninstall: simple-triton
    Found existing installation: simple-triton 0.1.dev418+g6eb2c6b.d20230920
    Uninstalling simple-triton-0.1.dev418+g6eb2c6b.d20230920:
      Successfully uninstalled simple-triton-0.1.dev418+g6eb2c6b.d20230920
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Processing /home/lac5440/glimr
  Installing build dependencies ... done
  Getting require

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.3 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.9/69.9 kB 46.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 37.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 28.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 75.9 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 17.2 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.9/468.9 kB 74.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 kB 23.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 32.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 43.5 MB/s eta 0:00:00
  Created wheel for glimr: filename=glimr-0.1.dev122+g815ca45-py3-none-any.whl size=21726 sha256=9c252cab01f51f2d35e86416163fe4fa2e3c236dda3cc48a9ee4b13709180ba0
  Stored in directory: /tmp/pip-ephem-wheel-cache-82q70tn9/wheels/bd/b6/4b/aa7d6a78c3ea85f3c561af5fa2328ebb6ad3866fa493f5645a
  Created wheel for gpustat: filename=gpustat-1.1.1-py3-none-any.whl size=26428 sha256=bb20c818d554a1590582debde104b102f2ae58cf06b882e200ad7bbb66a9e5aa
  Stored in directory: /home/lac5440/.cache/pip/wheels/ec/d7/80/a71ba35409

  Created wheel for mil: filename=mil-0.0.1-py3-none-any.whl size=48463 sha256=b9bec276b75723207dbbfe936bc5aabbb7e93c98263f156db12c10f4fc104cae
  Stored in directory: /tmp/pip-ephem-wheel-cache-ya5_ry_p/wheels/3f/7f/ff/37340b9860dc6a822d46598d60b983dcfda706371ceafcd137
Successfully built mil
  Attempting uninstall: mil
    Found existing installation: mil 0.0.1
    Uninstalling mil-0.0.1:
      Successfully uninstalled mil-0.0.1


## Run client with no GPUs

If running Triton and the client on the same machine, we want to stop the client tensorflow from consuming GPU resources. By default, TensorFlow maps nearly all available GPU memory.

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

assert len(tf.config.list_physical_devices("GPU")) == 0

2023-09-20 03:20:50.920156: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-09-20 03:20:50.971470: I tensorflow/core/platform/cpu_feature_guard.cc:183] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Create a histomics stream study

Parameters in this cell are for reading from the whole-slide image (magnification, tile size, tile overlap, mask file).

In [6]:
from mil.io.utils import study
import pooch

# slide parameters
batch = 64
magnification = 20
tile = 224
overlap = 0
chunk = 224
mask_threshold = 0.5

# download whole slide image and corresponding mask
wsi_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.svs",
    url="https://drive.google.com/uc?export=download&id=19agE_0cWY582szhOVxp9h3kozRfB4CvV&confirm=t&uuid=6f2d51e7-9366-4e98-abc7-4f77427dd02c&at=ALgDtswlqJJw1KU7P3Z1tZNcE01I:1679111148632",
    known_hash="d046f952759ff6987374786768fc588740eef1e54e4e295a684f3bd356c8528f",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)
mask_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.mask.png",
    url="https://drive.google.com/uc?export=download&id=17GOOHbL8Bo3933rdIui82akr7stbRfta",
    known_hash="bb657ead9fd3b8284db6ecc1ca8a1efa57a0e9fd73d2ea63ce6053fbd3d65171",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path), t=(tile, tile), chunk=(tile, tile), target=20, source="exact"
)

## Create a feature extraction model

The function `feature_extractor` can be used to create feature extraction models in the model repository. Note - this cell will take time as the model is downloaded, saved, and loaded into triton. The model is saved into the designated model respository that is mounted on the triton container. The `feature_extractor` function handles the expected directory structure and naming that differs from a typical TensorFlow savedmodel.

In [10]:
import numpy as np
from pprint import pprint
from simple_triton.feature_extraction import tf_extractor
from simple_triton.model import TritonModel

# model parameters
keras_name = "EfficientNetV2S"
model_name = f"{keras_name}.tensorflow"  # set model name

# create the model and capture output dimensionality
if not os.path.exists(os.path.join("~/models", model_name)):
    dimension_output = tf_extractor(
        "~/models", keras_name, model_name, input_shape=(tile, tile, 3), pooling="avg"
    )

2023-09-20 03:28:25.613378: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,56,56,256]
	 [[{{node inputs}}]]
2023-09-20 03:28:25.622894: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,56,56,1024]
	 [[{{node inputs}}]]
2023-09-20 03:28:25.645131: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,56,56,256]
	 [[{{node inputs}}

2023-09-20 03:28:26.051667: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,1024]
	 [[{{node inputs}}]]
2023-09-20 03:28:26.060833: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,4096]
	 [[{{node inputs}}]]
2023-09-20 03:28:26.083240: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,1024]
	 [[{{node inputs

2023-09-20 03:28:26.481941: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,1024]
	 [[{{node inputs}}]]
2023-09-20 03:28:26.491685: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,4096]
	 [[{{node inputs}}]]
2023-09-20 03:28:26.513287: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,1024]
	 [[{{node inputs

2023-09-20 03:28:32.535836: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,1024]
	 [[{{node inputs}}]]
2023-09-20 03:28:32.566538: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,4096]
	 [[{{node inputs}}]]
2023-09-20 03:28:32.628338: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,1024]
	 [[{{node inputs

2023-09-20 03:28:33.684990: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,4096]
	 [[{{node inputs}}]]
2023-09-20 03:28:33.746566: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,1024]
	 [[{{node inputs}}]]
2023-09-20 03:28:34.288135: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,4096]
	 [[{{node inputs

2023-09-20 03:28:35.493282: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,1024]
	 [[{{node inputs}}]]
2023-09-20 03:28:35.528155: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,14,14,4096]
	 [[{{node inputs}}]]
2023-09-20 03:28:35.646995: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?,7,7,2048]
	 [[{{node inputs}}

INFO:tensorflow:Assets written to: ~/models/ConvNeXtXLarge.tensorflow/1/model.savedmodel/assets


INFO:tensorflow:Assets written to: ~/models/ConvNeXtXLarge.tensorflow/1/model.savedmodel/assets


## Load the model using `TritonModel`

After creating the model, we load the model into Triton using the `TritonModel` class. This class contains methods for loading, unloading, and checking the status of models. To load the model we create a simple configuration with batch size 64, and allow Triton to generate the remaining configuration fields. By default it will load a single copy of the model on each system GPU, and will often automatically set optimizations like pinned memory.

In [11]:
# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server

# load tensorflow model - set maximum batch size
model = TritonModel(model_name, url)
model.load(config={"maxBatchSize": batch})
assert model.is_loaded()
pprint(model.get_config())

{'backend': 'tensorflow',
 'defaultModelFilename': 'model.savedmodel',
 'dynamicBatching': {'preferredBatchSize': [64]},
 'input': [{'dataType': 'TYPE_FP32',
            'dims': ['224', '224', '3'],
            'name': 'input_1'}],
 'instanceGroup': [{'count': 1,
                    'gpus': [0, 1, 2, 3, 4, 5, 6, 7],
                    'kind': 'KIND_GPU',
                    'name': 'ConvNeXtXLarge.tensorflow'}],
 'maxBatchSize': 64,
 'name': 'ConvNeXtXLarge.tensorflow',
 'optimization': {'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32',
             'dims': ['2048'],
             'name': 'layer_normalization'}],
 'platform': 'tensorflow_savedmodel',
 'versionPolicy': {'latest': {'numVersions': 1}}}


## Advanced configuration with `ConfigBuilder`

The `ConfigBuilder` class provides access to advanced configuration options like backend optimizations. Here, we create a duplicate model on each GPU (`count=2`) and convert the model to mixed precision to improve speed and memory usage. The model is reloaded using this advanced configuration.

In [12]:
from simple_triton.config import ConfigBuilder

# initialize builder with a basic configuration
builder = ConfigBuilder(model_name, config={"maxBatchSize": batch})

# increase the number of model instances per GPU to 2
builder.add_instance_group(count=2)

# add automatic mixed precision
builder.add_mixed_precision()

# re-load model with new config
model.load(config=builder.config)

# print config
pprint(model.get_config())

{'backend': 'tensorflow',
 'defaultModelFilename': 'model.savedmodel',
 'dynamicBatching': {'preferredBatchSize': [64]},
 'input': [{'dataType': 'TYPE_FP32',
            'dims': ['224', '224', '3'],
            'name': 'input_1'}],
 'instanceGroup': [{'count': 2,
                    'gpus': [0, 1, 2, 3, 4, 5, 6, 7],
                    'kind': 'KIND_GPU',
                    'name': 'ConvNeXtXLarge.tensorflow_0'}],
 'maxBatchSize': 64,
 'name': 'ConvNeXtXLarge.tensorflow',
 'optimization': {'executionAccelerators': {'gpuExecutionAccelerator': [{'name': 'auto_mixed_precision'}]},
                  'inputPinnedMemory': {'enable': True},
                  'outputPinnedMemory': {'enable': True}},
 'output': [{'dataType': 'TYPE_FP32',
             'dims': ['2048'],
             'name': 'layer_normalization'}],
 'platform': 'tensorflow_savedmodel',
 'versionPolicy': {'latest': {'numVersions': 1}}}


## Run the inference

Parameters here include the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [14]:
from simple_triton.feature_extraction import histomics_stream_inference
from simple_triton.utils import analyze
import time

# inference parameters
batch = 64
limit = 10  # limit on number of pending requests per worker
workers = 32  # total number of Submitter workers
verbose = True  # set verbose as False

# start timer
start = time.time()

# inference
features, tile_info, times, failures = histomics_stream_inference(
    study=hs_study,
    model_name=model_name,
    url=url,
    batch=batch,
    workers=workers,
    limit=limit,
)

# display elapsed time
print(f"Total elapsed time: {time.time()-start}")

# analyze performance
analyze(times)

Total elapsed time: 18.31273341178894
                             median    min    max
-------------------------  --------  -----  -----
total (sec)                    6.96   2.62  15.93
data loading (% total)        30.88   5.68  59.16
results return (% total)       0.10   0.02   0.26
in-process (% total)          69.02  40.65  94.30
completion (% in-process)     99.25  37.66  99.86
retrieval (% in-process)       0.08   0.00  42.44
other (% in-process)           0.68   0.11  40.82


## Write features to .tfr

In [15]:
from mil.io.reader import read_record, peek
from mil.io.writer import write_record
import tensorflow as tf

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record(
    "./triton.tfr", features, tile_info, labels, structured=False, precision=tf.float16
)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)

2023-09-20 03:31:41.084188: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]
2023-09-20 03:31:42.988711: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'cond/ScatterNd/Cast_1' with dtype int32 and shape [?]
	 [[{{node cond/ScatterNd/Cast_1}}]]
2023-09-20 03:31:42.988850: I tensorflow/core/common_runtime/executor.cc:1209] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'cond/ScatterNd/Cast_1' with dtype int32 an

(<tf.Tensor: shape=(4855, 2048), dtype=float16, numpy=
 array([[ 1.133e+00,  5.283e-03,  5.186e-01, ...,  5.977e+00, -2.006e+00,
          7.935e-02],
        [ 7.505e-01, -1.411e-01,  6.689e-01, ...,  4.742e+00, -2.055e+00,
         -1.622e-01],
        [ 6.548e-01, -1.537e-01,  4.536e-01, ...,  4.574e+00, -1.308e+00,
          6.519e-02],
        ...,
        [ 1.435e+00, -3.284e-01,  3.105e-01, ...,  5.805e+00, -1.308e+00,
          9.404e-01],
        [ 5.151e-01,  4.974e-02,  2.632e-01, ...,  4.555e+00, -3.221e+00,
          6.826e-01],
        [ 1.780e+00,  9.998e-02, -8.649e-02, ...,  2.328e+00, -2.232e+00,
         -3.530e-01]], dtype=float16)>,
 {'labels': <tf.Tensor: shape=(10,), dtype=float32, numpy=
  array([0.04655097, 0.00358522, 0.7985301 , 0.41142523, 0.8568527 ,
         0.47578222, 0.31158754, 0.28509724, 0.57508576, 0.8648176 ],
        dtype=float32)>},
 {'filename': <tf.Tensor: shape=(1,), dtype=string, numpy=
  array([b'/home/lac5440/.cache/pooch/wsi/TCGA-AN-A0G0-